# 📡 Gemini

**Python, Datos e Ingeniería de IA Aplicada**  
UTN Rosario — FRRO, Área de Extensión Universitaria — 2026

**Encuentro 3 · Bloque 3 — 60 minutos**

Docente: **Matías Barreto** — [matiasbarreto.com](https://matiasbarreto.com)

---

## Objetivo

Conectar los conceptos de Python que venimos trabajando (variables, funciones, listas, diccionarios, bucles) con la Gemini API de Google. La IA deja de ser algo abstracto y se convierte en una herramienta concreta que controlamos desde el código.

## Al terminar este bloque van a poder:

1. Instalar y configurar el SDK de Gemini en Python.
2. Usar variables de entorno para manejar la API key de forma segura.
3. Generar texto con el modelo `gemini-2.0-flash`.
4. Construir prompts dinámicos con variables de Python.
5. Automatizar tareas con listas, bucles y funciones que llaman a la API.
6. Analizar imágenes con capacidades multimodales de Gemini.

---

> **◈ Capítulo 3 de la cursada**
> En la clase 2 consumimos APIs de datos (Open-Meteo) y trabajamos con funciones, listas y archivos. Hoy damos un paso más: la API que vamos a consumir es un modelo de lenguaje. Eso cambia algo fundamental: la entrada ya no es una coordenada geográfica sino **lenguaje natural**, y la salida tampoco es JSON de datos meteorológicos sino **texto generado**.

---

## ◈ Microglosario

Antes de hacer la primera llamada, alineamos el vocabulario.

| Término | Qué es en lenguaje llano |
|---|---|
| **Prompt** | El texto que le enviamos al modelo como entrada. Es nuestra instrucción o pregunta. |
| **Respuesta (completion)** | El texto que el modelo genera como salida. |
| **Modelo** | El sistema de IA que procesa el prompt y genera la respuesta. Cada modelo tiene capacidades, velocidades y costos distintos. |
| **Token** | La unidad básica de procesamiento. No es exactamente una palabra; puede ser parte de una. Los modelos cuentan tokens para medir longitud y costo. |
| **API key** | La clave que identifica nuestra cuenta y autoriza las llamadas. Nunca la compartimos ni la escribimos en el código. |
| **Multimodal** | Un modelo que puede procesar más de un tipo de datos: texto, imágenes, audio, video. |

---

## 1. ◈ ¿Qué es un LLM y para qué sirve en código?

### Analogía

Cuando llamamos a la Gemini API desde Python, no "entramos a la cocina" a saber cómo funciona el modelo por dentro. Funcionamos igual que con un mozo: describimos lo que queremos, él lo trae. No importa cómo lo prepara la cocina.

Nosotros escribimos un **prompt** (el pedido). La API devuelve una **respuesta** (el plato). El modelo de lenguaje es la cocina: potente, compleja, y accesible a través de una interfaz simple.

### Dónde vive esto en el mundo real

Los LLMs son la base de productos que usamos a diario: asistentes de código, traducción automática, resumen de documentos, análisis de contratos, generación de reportes. Lo que los hace útiles para nosotros como programadores es que los podemos controlar desde código: automatizar, encadenar, parametrizar. Eso es lo que vamos a hacer hoy.

### ✎ Para pensar

- ¿Qué diferencia hay entre usar ChatGPT en el navegador y llamar a la Gemini API desde Python?
- ¿Qué tareas repetitivas que hacen en el trabajo o la facultad podrían automatizarse con un LLM?

---

## 2. ✦ Instalación y primera llamada

In [ ]:
# Instalamos el SDK oficial de Gemini y python-dotenv para las variables de entorno:
!uv pip install google-genai python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv

# load_dotenv() lee el archivo .env y carga cada variable en el entorno del proceso:
load_dotenv()

# Leemos la clave de la variable de entorno:
clave_api = os.getenv("GEMINI_API_KEY")

# Verificamos que esté cargada antes de seguir:
if clave_api:
    print("✓ API key cargada correctamente.")
else:
    print("✗ No se encontró GEMINI_API_KEY. Revisá el archivo .env")

#### Si están en Google Colab

En Colab no hay archivo `.env` local. Usen los Secrets de Colab:

1. Ícono de llave en el panel izquierdo
2. Agregar secreto: nombre `GEMINI_API_KEY`, valor: su clave
3. Activar el toggle "Acceso al notebook"
4. Reemplazar el bloque anterior por:

```python
from google.colab import userdata
clave_api = userdata.get('GEMINI_API_KEY')
```

In [ ]:
from google import genai

# Creamos el cliente con la clave que cargamos del entorno:
cliente = genai.Client(api_key=clave_api)

print("✓ Cliente de Gemini inicializado.")

In [ ]:
# Primera llamada al modelo.
# generate_content() recibe el nombre del modelo y el texto del prompt:
respuesta = cliente.models.generate_content(
    model="gemini-2.5-flash",
    contents="Escribí un haiku sobre los limones en primavera"
)

# La respuesta viene en respuesta.text:
print(respuesta.text)

In [ ]:
# El SDK no devuelve un string directamente: devuelve un objeto.
# El atributo .text es el acceso rápido al texto generado:
print("Tipo del objeto:", type(respuesta))
print()
print("Texto generado:")
print(respuesta.text)

### ✎ Para pensar

- ¿Qué pasa si cambian el prompt a "Escribí un haiku sobre Docker"?
- ¿Cómo cambiaría la respuesta si piden un poema de diez líneas en vez de un haiku?
- ¿Qué información del objeto `respuesta` les parece útil además del texto?

---

## 3. ◈ Variables de Python en los prompts

### Analogía

Un prompt hardcodeado es como un formulario impreso: sirve para un solo caso. Un prompt construido con f-strings es como un formulario digital: se completa con los datos del momento. La potencia real de programar con LLMs está en esta combinación.

### Dónde vive esto en el mundo real

Todos los productos de IA en producción generan prompts dinámicamente. Un asistente de atención al cliente construye un prompt diferente para cada ticket. Un sistema de resumen genera el prompt con el texto del documento que llega. Lo que van a ver hoy es el patrón básico que está debajo de todos esos sistemas.

In [ ]:
# Definimos variables con información que queremos incluir en el prompt:
nombre = "Otto Matic"

# Dividimos la edad del perro en pasos separados (una idea por línea):
edad_humana = 21
edad_en_anos_perro = 7
edad_perro = round(edad_humana / edad_en_anos_perro)

# Construimos el prompt con un f-string:
prompt = f"""Si {nombre} fuera un perro, tendría {edad_perro} años.
Describí en qué etapa de la vida estaría ese perro y qué podría implicar
en términos de nivel de energía, intereses y comportamiento.
Respondé en dos párrafos."""

print("Prompt construido:")
print(prompt)

In [ ]:
# Enviamos el prompt dinámico al modelo:
respuesta = cliente.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

In [ ]:
from IPython import display

display.Markdown(respuesta.text)

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Construí un prompt dinámico que use al menos tres variables:
#   1. Elegí un dominio: receta, descripción de producto, correo profesional
#   2. Definí las variables necesarias (nombre, ingrediente, tono, etc.)
#   3. Construí el prompt con f-string y envialo al modelo
#   4. Mostrá la respuesta
#
# Bonus: cambiá el valor de una variable y corré la celda de nuevo.
#        ¿Qué parte del código cambió? ¿Qué parte no tuvo que cambiar?
#
# ─────────────────────────────────────────────────────────────────────────────

---

## 4. ◈ Análisis de sentimiento

### Analogía

Un analista de mercado lee cientos de comentarios de clientes y los clasifica: positivo, negativo, neutro. Es trabajo manual, lento y subjetivo. Con un LLM, le pasamos el texto y las categorías posibles, y el modelo hace la clasificación. No entrenamos nada; solo diseñamos el prompt.

### Dónde vive esto en el mundo real

El análisis de sentimiento se usa en monitoreo de redes sociales, análisis de reseñas de productos, detección de tickets críticos en soporte al cliente, y evaluación de satisfacción en encuestas. Antes requería modelos especializados; hoy se resuelve con un buen prompt.

In [ ]:
# Texto a analizar:
tweet = """Thank you @AbiyAhmedAli for your warm welcome during my visit to Ethiopia.
I'm inspired by our insightful discussions on Ethiopia's development progress,
and I'm excited by the opportunity to continue supporting our partners
to track and accelerate progress in health, agriculture and financial inclusion."""

# Prompt que describe la tarea con precisión:
prompt_sentimiento = f"""Analizá el sentimiento del siguiente texto.
Respondé SOLO con una de estas opciones: POSITIVO, NEGATIVO o NEUTRO.
Después de la etiqueta, agregá una oración explicando por qué.

Texto: {tweet}"""

respuesta = cliente.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_sentimiento
)

display.Markdown(respuesta.text)

### ✎ Para pensar

- ¿Qué pasaría si quitáramos la instrucción "Respondé SOLO con una de estas opciones"? ¿Cómo cambiaría la respuesta?
- ¿Cómo usarían esta función para procesar una lista de 100 tweets de forma automática?

---

## 5. ✦ Funciones que encapsulan llamadas a la API

### Analogía

Cada vez que queremos obtener una respuesta del modelo, repetimos el mismo patrón: construir el prompt, llamar a `generate_content`, extraer `.text`. Una función encapsula ese patrón como una receta guardada: la escribimos una vez y la usamos cien veces.

### Dónde vive esto en el mundo real

Esto es exactamente lo que vimos en la clase 2: encapsular comportamiento repetido en funciones para que el código sea claro y fácil de mantener. En sistemas de producción, cada capacidad del LLM —traducir, resumir, clasificar— vive en su propia función. Si el modelo cambia, se actualiza en un solo lugar.

In [ ]:
def obtener_respuesta(prompt):
    """Envía un prompt al modelo y devuelve el texto de la respuesta."""

    # Llamamos a la API con el prompt recibido:
    respuesta = cliente.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    # Devolvemos solo el texto:
    return respuesta.text


print("✓ Función definida.")

In [ ]:
# Ahora podemos usar obtener_respuesta() en cualquier parte del código:
nombre = "Tommy"
papas = 4.75

# round() redondea el número de papas al entero más cercano:
papas_redondeadas = round(papas)

prompt = f"Escribí un pareado sobre mi amigo {nombre} que tiene como {papas_redondeadas} papas"

respuesta = obtener_respuesta(prompt)
display.Markdown(respuesta)

In [ ]:
# Podemos hacer funciones más especializadas que encapsulen toda la lógica del prompt:
def traducir(palabra, idioma_destino):
    """Traduce una palabra al idioma indicado usando Gemini."""

    # Construimos el prompt con instrucciones precisas:
    prompt = f"""Traducí la palabra "{palabra}" al {idioma_destino}.
    Respondé SOLO con la traducción, sin explicaciones ni puntuación extra."""

    # Usamos la función genérica que ya definimos:
    traduccion = obtener_respuesta(prompt)

    return traduccion.strip()


# Probamos la función:
resultado = traducir("hello", "checo")
print(f"'hello' en checo: {resultado}")

In [ ]:
# Otro ejemplo: una función que genera recetas con parámetros específicos:
ingrediente_secreto = "ajo"
numero_de_lineas = 5

prompt_receta = f"""Escribí una receta que use {ingrediente_secreto} como ingrediente principal.
La receta debe tener como máximo {round(numero_de_lineas)} pasos.
Sé conciso y claro."""

respuesta = obtener_respuesta(prompt_receta)
display.Markdown(respuesta)

### ✎ Para pensar

- ¿Qué ventaja tiene la función `traducir()` frente a llamar a `obtener_respuesta()` directamente con el texto del prompt?
- Si mañana cambiamos de `gemini-2.0-flash` a otro modelo, ¿cuántos lugares del código hay que modificar?

---

## 6. ◈ Listas y Gemini

Las listas de Python se pueden incluir directamente en los prompts. El modelo las entiende como enumeraciones.

In [ ]:
# Definimos una lista de nombres:
lista_amigos = ["Tommy", "Isabel", "Daniel"]

# Construimos un prompt que incluye la lista:
prompt_poemas = f"""
Escribí un poema de cumpleaños de cuatro líneas para cada uno de mis amigos: {lista_amigos}.
Los poemas deben estar inspirados en la primera letra del nombre de cada amigo.
Separalos con una línea en blanco.
"""

print("Prompt que vamos a enviar:")
print(prompt_poemas)

In [ ]:
# Enviamos el prompt al modelo:
respuesta = obtener_respuesta(prompt_poemas)
print(respuesta)

---

## 7. ✦ Automatización con bucles for

### Analogía

En la sección anterior mandamos toda la lista en un solo prompt. Eso es como darle a un mozo una lista de diez pedidos a la vez y esperar que los entregue todos en un plato. Un bucle `for` es diferente: es ir a la cocina una vez por pedido, traer una cosa, volver. Más control, respuestas más limpias.

### Dónde vive esto en el mundo real

Los pipelines de procesamiento de texto en producción procesan documento por documento, registro por registro. Así se puede manejar errores por ítem, registrar cuál falló, y controlar la velocidad para no exceder los límites de la API.

In [ ]:
# Lista de tareas a completar, en orden de prioridad.
# En Python se permiten listas de varias líneas:
lista_de_tareas = [
    "Redactá un breve correo electrónico a mi jefe explicando que voy a llegar tarde a la reunión de mañana.",
    "Escribí un poema de cumpleaños para Otto, celebrando sus 28 años.",
    "Escribí una reseña de 150 palabras de la película 'La llegada'."
]

print(f"✓ Lista de tareas: {len(lista_de_tareas)} ítems")

In [ ]:
# Primero probamos con una sola tarea (el primer ítem de la lista):
tarea = lista_de_tareas[0]

print("Tarea:", tarea)
print()

respuesta = obtener_respuesta(tarea)
display.Markdown(respuesta)

In [ ]:
# Ahora automatizamos: procesamos todas las tareas con un bucle for:
for tarea in lista_de_tareas:

    # Imprimimos la tarea actual para saber en qué estamos:
    print(f"\n{'='*60}")
    print(f"Tarea: {tarea}")
    print(f"{'='*60}")

    # Enviamos la tarea al modelo y mostramos la respuesta:
    respuesta = obtener_respuesta(tarea)
    print(respuesta)

In [ ]:
# Otro ejemplo: generar descripciones promocionales para sabores de helado:
sabores_helados = [
    "Vainilla",
    "Chocolate",
    "Pistacho",
    "Menta granizada"
]

# Procesamos cada sabor por separado:
for sabor in sabores_helados:

    # Construimos un prompt específico para cada sabor:
    prompt = f"""Para el sabor de helado "{sabor}", escribí una descripción
    cautivadora en español de UNA SOLA LÍNEA para uso promocional."""

    respuesta = obtener_respuesta(prompt)

    # Mostramos el resultado con formato claro:
    print(f"◇ {sabor}: {respuesta.strip()}")

### ✎ Para pensar

- ¿Qué diferencia hace incluir `sabor` en el prompt de cada iteración versus enviarlo una vez en el prompt de la lista completa?
- ¿En qué situaciones sería mejor una única llamada con toda la lista? ¿En cuáles sería mejor el bucle?
- ¿Cómo modificarían el bucle para guardar todas las respuestas en una nueva lista?

---

## 8. ◈ Diccionarios como contexto del prompt

### Analogía

Un diccionario de Python es como la ficha de un cliente en una base de datos: cada campo tiene un nombre y un valor. Cuando incluimos esa ficha en el prompt, el modelo puede razonar sobre toda la información estructurada a la vez, igual que lo haría un analista humano.

### Dónde vive esto en el mundo real

Los sistemas de recomendación, personalización y asistencia construyen el prompt a partir de un perfil del usuario. Cuanto más rico el contexto que le pasamos al modelo, más pertinentes son sus respuestas.

In [ ]:
# Perfil de preferencias alimentarias de Tommy:
preferencias_tommy = {
    "restricciones_dietarias": "vegetariano",
    "ingredientes_favoritos": ["tofu", "aceitunas"],
    "nivel_de_experiencia": "intermedio",
    "nivel_maximo_de_picante": 6
}

print("Perfil de Tommy:")
for clave, valor in preferencias_tommy.items():
    print(f"  {clave}: {valor}")

In [ ]:
# Usamos los valores del diccionario para construir un prompt personalizado:
prompt_receta_tommy = f"""Sugerí una receta que incluya los siguientes ingredientes:
{preferencias_tommy["ingredientes_favoritos"]}.

La receta debe cumplir estas condiciones:
- Restricción dietaria: {preferencias_tommy["restricciones_dietarias"]}
- Nivel de dificultad: {preferencias_tommy["nivel_de_experiencia"]}
- Picante máximo (escala 1-10): {preferencias_tommy["nivel_maximo_de_picante"]}

Respondé con el nombre del plato y una descripción de dos oraciones."""

respuesta = obtener_respuesta(prompt_receta_tommy)
display.Markdown(respuesta)

In [ ]:
# Refinamos el prompt con información adicional — las especias disponibles:
especias_disponibles = ["comino", "cúrcuma", "orégano", "pimentón"]

# Agregamos la restricción de especias al prompt anterior:
prompt_receta_refinada = f"""Sugerí una receta que incluya los siguientes ingredientes:
{preferencias_tommy["ingredientes_favoritos"]}.

La receta debe cumplir estas condiciones:
- Restricción dietaria: {preferencias_tommy["restricciones_dietarias"]}
- Nivel de dificultad: {preferencias_tommy["nivel_de_experiencia"]}
- Picante máximo (escala 1-10): {preferencias_tommy["nivel_maximo_de_picante"]}
- Las especias usadas deben pertenecer SOLO a esta lista: {especias_disponibles}

Respondé con el nombre del plato y una descripción de dos oraciones."""

respuesta = obtener_respuesta(prompt_receta_refinada)
display.Markdown(respuesta)

### ✎ Para pensar

- ¿Qué pasa con la respuesta cuando agregamos la restricción de especias? ¿Se ve el cambio?
- ¿Cómo modificarían el código para procesar perfiles de 10 usuarios distintos de forma automática?

---

## 9. ✦ ChatBot con historial de conversación

### Analogía

Imaginen hablar con alguien que sufre amnesia total entre cada frase que les dicen. Cada vez que hablan, esa persona no recuerda nada de lo anterior. Así funciona un LLM sin historial. La solución es simple: antes de cada respuesta, le leemos en voz alta todo lo que se dijo antes. El modelo "recuerda" porque nosotros le leemos el registro.

### Dónde vive esto en el mundo real

Todos los chatbots de producción implementan este patrón. La diferencia entre una conversación coherente y una que "olvida" el contexto es exactamente si el historial se envía o no en cada llamada.

In [ ]:
# Iniciamos el historial de la conversación como una lista vacía:
historial_conversacion = []

print("Chatbot iniciado. Escribí 'salir' para terminar.")
print()

while True:
    # Esperamos el input del usuario:
    entrada_usuario = input("Usuario: ")

    # Si el usuario escribe 'salir', terminamos el bucle:
    if entrada_usuario.lower() == "salir":
        print("Conversación finalizada.")
        break

    # Agregamos el mensaje del usuario al historial:
    historial_conversacion.append({
        "role": "user",
        "parts": [{"text": entrada_usuario}]
    })

    # Enviamos el historial completo al modelo:
    respuesta = cliente.models.generate_content(
        model="gemini-2.5-flash",
        contents=historial_conversacion
    )

    # Extraemos el texto de la respuesta:
    texto_respuesta = respuesta.text

    print(f"Asistente: {texto_respuesta}")
    print()

    # Agregamos la respuesta del modelo al historial:
    historial_conversacion.append({
        "role": "model",
        "parts": [{"text": texto_respuesta}]
    })

---

## 🐛 Laboratorio de Rotura

El código de abajo *parece* un chatbot con memoria. Antes de correrlo, predicción: ¿el modelo va a recordar lo que se dijo en turnos anteriores?

```python
while True:
    entrada_usuario = input("Usuario: ")

    if entrada_usuario.lower() == "salir":
        break

    respuesta = cliente.models.generate_content(
        model="gemini-2.0-flash",
        contents=entrada_usuario  # ← solo el mensaje actual, sin historial
    )

    print(f"Asistente: {respuesta.text}")
```

Probá decirle al modelo tu nombre en el primer mensaje y preguntarle en el segundo: "¿Cómo me llamo?"

Antes de avanzar:

- ¿Qué respondió cuando le preguntaste tu nombre?
- ¿Por qué no lo recuerda, si acabás de decírselo?
- ¿Qué elemento específico del código de la sección anterior resuelve este problema?

**No mires la resolución todavía. Primero la hipótesis.**

### Resolución del Laboratorio de Rotura

El modelo no tiene estado entre llamadas. Cada `generate_content()` recibe solo lo que le pasamos en `contents`. Si pasamos únicamente el mensaje actual, el modelo no sabe nada de lo que se dijo antes.

La solución es enviar el **historial completo** en cada llamada:

```python
# ✗ Sin memoria — solo el mensaje actual:
contents=entrada_usuario

# ✓ Con memoria — la lista completa de la conversación:
contents=historial_conversacion
```

Esta es una diferencia conceptual fundamental: **el modelo es stateless** (sin estado). La ilusión de memoria la construimos nosotros, acumulando y enviando el historial.

---

## 10. ◈ Visión — análisis de imágenes

### Analogía

Un texto es una sola dimensión: palabras en secuencia. Una imagen es dos dimensiones: píxeles en espacio. Para enviarle una imagen a un modelo de lenguaje, necesitamos convertirla a texto. Base64 es esa conversión: toma los bytes de la imagen y los representa como caracteres. El modelo los recibe y los "entiende" como imagen.

### Dónde vive esto en el mundo real

Gemini es un modelo **multimodal**: puede procesar texto, imágenes, audio y video en la misma llamada. Esto abre aplicaciones que antes requerían modelos separados: analizar radiografías, verificar documentos, moderar contenido visual, describir productos para e-commerce.

In [ ]:
import base64
import httpx

# URL de una imagen pública:
url_imagen = "https://content.nationalgeographic.com.es/medio/2024/05/31/macho-jin-xi-en-el-zoo-de-madrid_355544e5_240531135148_800x800.jpg"
tipo_media = "image/jpeg"

# Descargamos la imagen:
respuesta_imagen = httpx.get(
    url_imagen,
    follow_redirects=True,
    headers={"User-Agent": "Mozilla/5.0"}
)

# Verificamos que la descarga haya funcionado:
print(f"Status: {respuesta_imagen.status_code}")
print(f"Tipo de contenido: {respuesta_imagen.headers.get('content-type')}")

respuesta_imagen.raise_for_status()

# Codificamos los bytes de la imagen en Base64:
imagen_en_base64 = base64.b64encode(respuesta_imagen.content)

# Convertimos los bytes Base64 a string:
imagen_string = imagen_en_base64.decode("utf-8")

# Verificamos el resultado:
total_caracteres = len(imagen_string)
print(f"Imagen codificada: {total_caracteres:,} caracteres en Base64")
print(f"Primeros caracteres: {imagen_string[:50]}...")

In [ ]:
# Enviamos la imagen al modelo con un prompt de descripción:
mensajes = [
    {
        "role": "user",
        "parts": [
            {
                "inline_data": {
                    "mime_type": tipo_media,
                    "data": imagen_string
                }
            },
            {
                "text": "Describí la imagen en dos párrafos."
            }
        ]
    }
]

respuesta = cliente.models.generate_content(
    model="gemini-2.5-flash",
    contents=mensajes
)

print(respuesta.text)

In [ ]:
# Cambiamos el prompt para pedir un análisis diferente de la misma imagen:
mensajes_paleta = [
    {
        "role": "user",
        "parts": [
            {
                "inline_data": {
                    "mime_type": tipo_media,
                    "data": imagen_string
                }
            },
            {
                "text": "Describí el tipo de paleta de color de la imagen y, si es posible, el sentimiento o estado de ánimo que transmite."
            }
        ]
    }
]

respuesta = cliente.models.generate_content(
    model="gemini-2.5-flash",
    contents=mensajes_paleta
)

print(respuesta.text)

### ✎ Para pensar

- ¿Qué diferencia hay entre la respuesta de descripción general y la de análisis de paleta? ¿Qué papel cumple el prompt en esa diferencia?
- ¿Cómo usarían esta capacidad para procesar automáticamente las fotos de productos de un catálogo?

---

> **✎ Mentor reversible (bonus).** Hasta acá fueron quienes aprenden. Ahora denlo vuelta: tomen la función `obtener_respuesta()` de la sección 5 y escriban, arriba de cada línea, un comentario que se la explique a alguien que nunca usó una API en su vida. Si esa persona imaginaria lo entiende, es porque ustedes ya lo entendieron de verdad.

---

## ⛰️ Diario de Navegación

Cerramos mirando hacia adentro, no hacia el código. Respondan en una o dos líneas, para ustedes:

1. ¿Qué concepto de hoy les costó más encajar en la cabeza? ¿Por qué creen que se resistió?
2. Si tuvieran que explicarle este cuaderno a un colega en lo que dura un viaje en ascensor, ¿qué le dirían?

No hay respuesta correcta. Lo que escriban es el mapa de su propio recorrido.